# Vegetation segmentation: UNet on RGB + NDVI (+ shade)

Tiles are `(6, 250, 250)` uint8 arrays: channels `R, G, B, NDVI, shade, label`.
Label is `0 = background`, `1 = tall`, `2 = flat`.

A from-scratch `smp.Unet` is trained with Lightning, AdamW, kornia flip augmentation,
early stopping on the validation F1, then the best checkpoint is evaluated on the test split.

In [ ]:
import glob
import os
from functools import partial

import kornia.augmentation as K
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, jaccard_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.classification import MulticlassF1Score, MulticlassJaccardIndex

CLASS_NAMES = ["background", "tall", "flat"]  # label values 0, 1, 2

## Config

Flip `USE_SHADE` / `USE_AUG` to toggle the shade channel and augmentation.

In [ ]:
DATA = "data/dataset"
USE_SHADE = True   # True -> 5 input channels (adds shade), False -> 4 (R,G,B,NDVI)
USE_AUG = True     # kornia horizontal/vertical flips during training
EPOCHS = 50
BATCH_SIZE = 16
LR = 1e-3
NUM_WORKERS = 4
OUT = "outputs"

os.makedirs(OUT, exist_ok=True)
IN_CHANNELS = 5 if USE_SHADE else 4

## Dataset

In [ ]:
class NpyDataset(Dataset):
    """Tiles of shape (6, H, W): R, G, B, NDVI, shade, label."""

    def __init__(self, split_dir, use_shade=True):
        self.files = sorted(glob.glob(os.path.join(split_dir, "*.npy")))
        self.channels = 5 if use_shade else 4

    def __len__(self):
        return len(self.files)

    def __getitem__(self, index):
        tile = np.load(self.files[index]).astype(np.float32)
        image = torch.from_numpy(tile[: self.channels] / 255.0)
        label = torch.from_numpy(tile[5]).long()
        return image, label

## Model

mIoU (`MulticlassJaccardIndex`) and F1 (`MulticlassF1Score`) are the tracked metrics; F1 is the early-stopping target.

In [ ]:
class SegModel(L.LightningModule):
    def __init__(self, in_channels, num_classes=3, lr=1e-3, use_aug=True):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        self.use_aug = use_aug
        self.net = smp.Unet(
            encoder_name="resnet18",
            encoder_weights=None,
            in_channels=in_channels,
            classes=num_classes,
        )
        self.loss = nn.CrossEntropyLoss()
        self.aug = K.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            data_keys=["input", "mask"],
        )
        self.train_f1 = MulticlassF1Score(num_classes, average="micro")
        self.val_f1 = MulticlassF1Score(num_classes, average="micro")
        self.val_miou = MulticlassJaccardIndex(num_classes, average="micro")
        self.train_history = []
        self.val_history = []

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, _):
        image, label = batch
        if self.use_aug:
            image, mask = self.aug(image, label.unsqueeze(1).float())
            label = mask.squeeze(1).long()
        logits = self(image)
        loss = self.loss(logits, label)
        self.train_f1.update(logits.argmax(1), label)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self.train_history.append(self.train_f1.compute().item())
        self.train_f1.reset()

    def validation_step(self, batch, _):
        image, label = batch
        logits = self(image)
        self.val_f1.update(logits.argmax(1), label)
        self.val_miou.update(logits.argmax(1), label)
        self.log("val_loss", self.loss(logits, label), prog_bar=True)

    def on_validation_epoch_end(self):
        f1 = self.val_f1.compute().item()
        self.log("val_f1", f1, prog_bar=True)
        self.log("val_miou", self.val_miou.compute().item(), prog_bar=True)
        self.val_history.append(f1)
        self.val_f1.reset()
        self.val_miou.reset()

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)

## Data loaders

In [ ]:
def loader(split_dir, use_shade, batch_size, num_workers, shuffle):
    dataset = NpyDataset(split_dir, use_shade)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )


make = partial(loader, use_shade=USE_SHADE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
train_loader = make(os.path.join(DATA, "train"), shuffle=True)
val_loader = make(os.path.join(DATA, "val"), shuffle=False)
test_loader = make(os.path.join(DATA, "test"), shuffle=False)
len(train_loader.dataset), len(val_loader.dataset), len(test_loader.dataset)

## Train

Early stopping watches `val_f1` with patience 10; the best checkpoint is saved.

In [ ]:
model = SegModel(in_channels=IN_CHANNELS, lr=LR, use_aug=USE_AUG)
checkpoint = L.pytorch.callbacks.ModelCheckpoint(monitor="val_f1", mode="max", dirpath=OUT, filename="best")
early_stop = L.pytorch.callbacks.EarlyStopping(monitor="val_f1", mode="max", patience=10)
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    callbacks=[checkpoint, early_stop],
    num_sanity_val_steps=0,
    log_every_n_steps=10,
)
trainer.fit(model, train_loader, val_loader)

## Train vs validation F1

In [ ]:
epochs = range(1, len(model.train_history) + 1)
plt.figure()
plt.plot(epochs, model.train_history, marker="o", label="train F1")
plt.plot(epochs, model.val_history, marker="o", label="val F1")
plt.xlabel("epoch")
plt.ylabel("F1")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(OUT, "f1_curve.png"), bbox_inches="tight")
plt.show()

## Evaluate best checkpoint on test

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
best = SegModel.load_from_checkpoint(checkpoint.best_model_path).eval().to(device)

preds, targets = [], []
with torch.no_grad():
    for image, label in test_loader:
        logits = best(image.to(device))
        preds.append(logits.argmax(1).cpu().numpy().ravel())
        targets.append(label.numpy().ravel())
preds = np.concatenate(preds)
targets = np.concatenate(targets)

labels = list(range(len(CLASS_NAMES)))
print(classification_report(targets, preds, labels=labels, target_names=CLASS_NAMES, digits=4))
print("macro F1 :", round(f1_score(targets, preds, labels=labels, average="macro", zero_division=0), 4))
print("mIoU     :", round(jaccard_score(targets, preds, labels=labels, average="macro", zero_division=0), 4))
per_class_iou = jaccard_score(targets, preds, labels=labels, average=None, zero_division=0)
print("per-class IoU:", dict(zip(CLASS_NAMES, per_class_iou.round(4))))

## Confusion matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(targets, preds, display_labels=CLASS_NAMES, normalize="true")
plt.title("Test confusion matrix (row-normalised)")
plt.show()

## Test tiles: image / mask / prediction

In [ ]:
images, masks = next(iter(test_loader))
with torch.no_grad():
    grid_preds = best(images.to(device)).argmax(1).cpu()

n = 3
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
for row in range(n):
    rgb = images[row, :3].permute(1, 2, 0).numpy()
    for ax, (data, title) in zip(axes[row], [(rgb, "image"), (masks[row], "mask"), (grid_preds[row], "prediction")]):
        ax.imshow(data, vmin=0, vmax=len(CLASS_NAMES) - 1)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()